# Start Here: Digifly Phase 2

Run the cell below. It starts the Docker runtime when needed and opens the Phase 2 Workbench in JupyterLab.

Requirements: Docker Desktop must be installed and running. No PowerShell commands are required.


In [ ]:
from __future__ import annotations

import os
import subprocess
import time
import urllib.request
import webbrowser
from pathlib import Path

from IPython.display import HTML, Javascript, display

PORT = int(os.environ.get("DIGIFLY_JUPYTER_PORT", "8888"))
WORKBENCH_PATH = "/lab/tree/Phase%202/notebooks/Digifly_Phase2_Workbench.ipynb"
WORKBENCH_URL = f"http://localhost:{PORT}{WORKBENCH_PATH}"


def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "docker-compose.yml").exists():
            return candidate
    raise RuntimeError("Could not find docker-compose.yml. Open this notebook from the Digifly Public repo.")


def _inside_docker() -> bool:
    return Path("/.dockerenv").exists() or os.environ.get("DIGIFLY_WORKSPACE") == "/workspace"


def _wait_for_jupyter(timeout_s: int = 240) -> bool:
    api_url = f"http://localhost:{PORT}/api"
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(api_url, timeout=2) as response:
                if 200 <= int(response.status) < 500:
                    return True
        except Exception:
            time.sleep(2)
    return False


if _inside_docker():
    display(HTML("<b>Docker runtime is already active.</b> Opening the Phase 2 Workbench..."))
    display(Javascript(f"window.open('{WORKBENCH_PATH}', '_blank');"))
    display(HTML(f'If a new tab did not open, <a href="{WORKBENCH_PATH}" target="_blank">open the Phase 2 Workbench</a>.'))
else:
    repo = _repo_root()
    print("Repo:", repo)
    print("Starting Docker runtime...")
    subprocess.run(["docker", "info"], cwd=repo, check=True)
    subprocess.run(["docker", "compose", "up", "--build", "-d", "phase2-jupyter"], cwd=repo, check=True)
    print("Waiting for JupyterLab...")
    if not _wait_for_jupyter():
        raise RuntimeError("JupyterLab did not respond. Run docker compose logs phase2-jupyter for details.")
    print("Opening Phase 2 Workbench:", WORKBENCH_URL)
    webbrowser.open(WORKBENCH_URL)
    display(HTML(f'<b>Docker is running.</b> <a href="{WORKBENCH_URL}" target="_blank">Open the Phase 2 Workbench</a>.'))
